In [2]:
import yaml
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import text


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Leer credenciales del arhivo YAML
with open("credenciales.yaml", "r") as file:
     config = yaml.safe_load(file)

db = config["mysql"]

# Crear conexión
# Su función es crear un Engine, que es el objeto que SQLAlchemy utiliza para comunicarse con la base de datos.
engine = create_engine(                                     
    f"mysql+pymysql://{db['user']}:{db['password']}@"
    f"{db['host']}:{db['port']}/{db['database']}"
)

##### ANALISIS DE TEXTO

##### 5-1 Introduccion

Se utilizaran datos de reseñas de un parque de diversiones. El objetivo es aprender herramientas para limpieza y análisis básico de texto. 


##### 5-2 Qué es el analisis de texto

Los datos de tipo texto pueden ser: 

$\cdot$ Estructurados: 		Datos de formularios por ejemplo

$\cdot$ No estructurados:	Datos extraídos directamente de libros, revistas, páginas web.

$\cdot$ Semi estructurados:	Tienen parte estructurada y no estructurada como los correos electronicos.


Hay 2 tipos de análisis de texto:

$\cdot$ Cualitativo:	Leer el texto y hacer un resumen del contenido.

$\cdot$ Cuantitativo:	Categorización, extracción de información. SQL nos servira para este análisis.

Metas:

$\cdot$ Extracción de texto: Extraer el numero de estrellas que dejo una reseña.

$\cdot$ Categorización: Categorizar las reseñas.

$\cdot$ Análisis de sentimiento: Si al reseña fue positiva, negativa o neutra. SQL no es la mejor opción para realizar este análisis.

##### 5-3 Por que sí usar SQL

Si se cuenta con una gran cantidad de datos y están disponibles directamente en una base relacional. Es una herramienta adecuada para tareas de conteo
o frecuencias, busqueda de palabras clave, limpiar y estructurar datos.

##### 5-4 SQL no es bueno para trabajar con texto

Si la base es muy pequeña, si la consultas requieren ser ser ejecutadas en el menor tiempo posible hay alternativas como Elastic Search, 
si el objetivo es leer todo el texto y entregar un resumen.

##### 5-5 Caracteristicas del Texto


In [21]:
# Mostrar las primeras 5 filas de la tabla disneyrew del esquema analisistexto

query = """
select
    * 
from
    analisistexto.disneyrew
limit
    5;
"""
df = pd.read_sql_query(text(query), engine)

#Con esta sentencia with podemos mostrar todas las columnas y filas del DataFrame sin truncar el contenido de las celdas. Esto es útil cuando queremos inspeccionar los datos completos, especialmente si algunas columnas contienen texto largo o si queremos ver todas las columnas disponibles en el DataFrame.
with pd.option_context(
    'display.max_colwidth', None,
    'display.max_columns', None):
    display(df.head())

,disneyrew_id,Information
0,1,"Rating: 4, Year_Month: 2019-4, Reviewer_Location: Australia, Review_Text: If you've ever been to Disneyland anywhere you'll find Disneyland Hong Kong very similar in the layout when you walk into main street! It has a very familiar feel. One of the rides its a Small World is absolutely fabulous and worth doing. The day we visited was fairly hot and relatively busy but the queues moved fairly well. Branch: Disneyland_HongKong"
1,2,"Rating: 4, Year_Month: 2019-5, Reviewer_Location: Philippines, Review_Text: Its been a while since d last time we visit HK Disneyland .. Yet, this time we only stay in Tomorrowland .. AKA Marvel land!Now they have Iron Man Experience n d Newly open Ant Man n d Wasp!!Ironman .. Great feature n so Exciting, especially d whole scenery of HK (HK central area to Kowloon)!Antman .. Changed by previous Buzz lightyear! More or less d same, but I'm expecting to have something most!!However, my boys like it!!Space Mountain .. Turns into Star Wars!! This 1 is Great!!!For cast members (staffs) .. Felt bit MINUS point from before!!! Just dun feel like its a Disney brand!! Seems more local like Ocean Park or even worst!!They got no SMILING face, but just wanna u to enter n attraction n leave!!Hello this is supposed to be Happiest Place on Earth brand!! But, just really Dont feel it!!Bakery in Main Street now have more attractive delicacies n Disney theme sweets .. These are Good Points!!Last, they also have Starbucks now inside the theme park!! Branch: Disneyland_HongKong"
2,3,"Rating: 4, Year_Month: 2019-4, Reviewer_Location: United Arab Emirates, Review_Text: Thanks God it wasn t too hot or too humid when I was visiting the park otherwise it would be a big issue (there is not a lot of shade).I have arrived around 10:30am and left at 6pm. Unfortunately I didn t last until evening parade, but 8.5 hours was too much for me.There is plenty to do and everyone will find something interesting for themselves to enjoy.It wasn t extremely busy and the longest time I had to queue for certain attractions was 45 minutes (which is really not that bad).Although I had an amazing time, I felt a bit underwhelmed with choice of rides and attractions. The park itself is quite small (I was really expecting something grand even the main castle which was closed by the way was quite small).The food options are good, few coffee shops (including Starbucks) and plenty of gift shops. There was no issue with toilets as they are everywhere.All together it was a great day out and I really enjoyed it. Branch: Disneyland_HongKong"
3,4,"Rating: 4, Year_Month: 2019-4, Reviewer_Location: Australia, Review_Text: HK Disneyland is a great compact park. Unfortunately there is quite a bit of maintenance work going on at present so a number of areas are closed off (including the famous castle) If you go midweek, it is not too crowded and certainly no where near as bus as LA Disneyland. We did notice on this visit that prices for food, drinks etc have really gone through the roof so be prepared to pay top dollar for snacks (and avoid the souvenir shops if you can) Regardless, kids will love it. Branch: Disneyland_HongKong"
4,5,"Rating: 4, Year_Month: 2019-4, Reviewer_Location: United Kingdom, Review_Text: the location is not in the city, took around 1 hour from Kowlon, my kids like disneyland so much, everything is fine. but its really crowded and hot in Hong Kong Branch: Disneyland_HongKong"


In [22]:
# Obtener el numero de caracteres de la columna information */

query = """
select
    length(information) as tamaño
from
    analisistexto.disneyrew
limit
    5;
"""
df = pd.read_sql_query(text(query), engine)
with pd.option_context(
    'display.max_colwidth', None,
    'display.max_columns', None):
    display(df.head(10))

,tamaño
0,430
1,1074
2,1051
3,587
4,270


In [25]:
# Agrupar por el numero de caracteres que tienen las reseñas y hacer un conteo

query = """
select
    length(information) as tamaño,
    count(disneyrew_id)
from
    analisistexto.disneyrew
group by
    1
order by
    1 desc;
"""
df = pd.read_sql_query(text(query), engine)
with pd.option_context(
    'display.max_colwidth', None,
    'display.max_columns', None):
    display(df.head())

,tamaño,count(disneyrew_id)
0,20860,1
1,18564,1
2,18443,1
3,17466,1
4,17074,1


##### 5-6 Text Parsing 1


In [29]:
# Formula Left y Right para extraer caracteres de una cadena de texto

query = """
select
     left("Reseñas de Disney",7)  as izquierda,
     right("Reseñas de Disney",6) as derecha; 
"""
df = pd.read_sql_query(text(query), engine)
df.head()

,izquierda,derecha
0,Reseñas,Disney


In [31]:
# Extraer los primeros 6 caracteres de la columna de reseñas y contar cuantas veces se repite cada cadena */

query="""
select
    left(information,6) as izquierda,
    count(disneyrew_id)
from
    analisistexto.disneyrew
group by 1;
"""
df = pd.read_sql_query(text(query), engine)
with pd.option_context(
    'display.max_colwidth', None,
    'display.max_columns', None):
    display(df.head())

,izquierda,count(disneyrew_id)
0,Rating,42656


In [32]:
# Usar la función right y left para obtener la fecha en que se hizo el comentario. Notemos que faltaría limpiar la fecha.

query="""
select
    right(left(information,30),7) as fecha
from
    analisistexto.disneyrew
group by
    1
order by
    1;
"""
df = pd.read_sql_query(text(query), engine)
with pd.option_context(
    'display.max_colwidth', None,
    'display.max_columns', None):
    display(df.head())

,fecha
0,2010-10
1,2010-11
2,2010-12
3,"2010-3,"
4,"2010-4,"


In [37]:
# Función substring_index */
# Esta funcion busca una cadena especificada por el usuario sobre una columa de tipo string y devuelve la cadena antes de ese string o despues de ese string. */
# Si la cadena "un ejemplo" se repite varias veces en el string, la funcion tomara en cuenta solo la primera vez que aparezca */

query="""
select
    substring_index("Esto es un ejemplo de texto","un ejemplo", 1) as Antes, 
    substring_index("Esto es un ejemplo de texto","un ejemplo",-1) as Despues;
"""
df = pd.read_sql_query(text(query), engine)
with pd.option_context(
    'display.max_colwidth', None,
    'display.max_columns', None):
    display(df.head())

,Antes,Despues
0,Esto es,de texto


In [43]:
# Usar la funcion substring para extraer la fecha de la columna Infomrmation. Realizar un conteo de cuantas reseñas se hicieron por mes. Notemos que la fecha no esta en formato de fecha, sino como texto.

query = """
select
    substring_index(substring_index(information,"Year_Month: ",-1),",",1) as fecha,
    count(disneyrew_id)
from
    analisistexto.disneyrew
group by
    1
order by
    1;
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,fecha,count(disneyrew_id)
0,2010-10,21
1,2010-11,27
2,2010-12,56
3,2010-3,2
4,2010-4,1
5,2010-5,4
6,2010-6,8
7,2010-7,7
8,2010-8,7
9,2010-9,10


##### 5-7 Text Parsing 2

In [47]:
# Usando la consulta anterior obtener el año

query = """
select
    substring_index(substring_index(substring_index(information,"Year_Month: ",-1),",",1),"-",1) as año,
    count(disneyrew_id) as conteo
from
    analisistexto.disneyrew
group by
    1
order by
    1;
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)


,año,conteo
0,2010,143
1,2011,1984
2,2012,4342
3,2013,4717
4,2014,5301
5,2015,6979
6,2016,6599
7,2017,5195
8,2018,3997
9,2019,786


In [51]:
# Obtener el año y el mes

query="""
select
     substring_index(substring_index(substring_index(information,"Year_Month: ",-1),",",1),"-",1) as año,
     substring_index(substring_index(substring_index(information,"Year_Month: ",-1),",",1),"-",-1) as mes,
     count(disneyrew_id) as conteo
from
     analisistexto.disneyrew
group by
     1,2
order by
     1;
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,año,mes,conteo
0,2010,10,21
1,2010,11,27
2,2010,12,56
3,2010,3,2
4,2010,4,1
5,2010,5,4
6,2010,6,8
7,2010,7,7
8,2010,8,7
9,2010,9,10


In [56]:
# Usar la funcion substring_index para obtener rating, fecha, pais de origen.

query="""
select
     substring_index(substring_index(information,", Year_Month: ",1),"Rating: ",-1) as rating,
     substring_index(substring_index(information,", Reviewer_Location: ",1),", Year_Month: ",-1) as fecha,
     substring_index(substring_index(substring_index(information,", Reviewer_Location: ",1),", Year_Month: ",-1),"-",1) as año,
     substring_index(substring_index(substring_index(information,", Reviewer_Location: ",1),", Year_Month: ",-1),"-",-1) as mes,
     substring_index(substring_index(information,", Review_Text: ",1),", Reviewer_Location: ",-1) as pais_origen,
     substring_index(substring_index(information,", Branch: ",1),", Review_Text: ",-1) as comentario,
     substring_index(Information," Branch: ",-1) as parque
from
     analisistexto.disneyrew;
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,rating,fecha,año,mes,pais_origen,comentario,parque
0,4,2019-4,2019,4,Australia,If you've ever been to Disneyland anywhere you...,Disneyland_HongKong
1,4,2019-5,2019,5,Philippines,Its been a while since d last time we visit HK...,Disneyland_HongKong
2,4,2019-4,2019,4,United Arab Emirates,Thanks God it wasn t too hot or too humid wh...,Disneyland_HongKong
3,4,2019-4,2019,4,Australia,HK Disneyland is a great compact park. Unfortu...,Disneyland_HongKong
4,4,2019-4,2019,4,United Kingdom,"the location is not in the city, took around 1...",Disneyland_HongKong
5,3,2019-4,2019,4,Singapore,"Have been to Disney World, Disneyland Anaheim ...",Disneyland_HongKong
6,5,2019-4,2019,4,India,Great place! Your day will go by and you won't...,Disneyland_HongKong
7,3,2019-3,2019,3,Malaysia,Think of it as an intro to Disney magic for th...,Disneyland_HongKong
8,2,2019-4,2019,4,Australia,"Feel so let down with this place,the Disneylan...",Disneyland_HongKong
9,5,2019-3,2019,3,India,I can go on talking about Disneyland. Whatever...,Disneyland_HongKong


##### 5-8 Transformacion de Texto

In [58]:
# Funciones Upper, Lower, Trim

query = """
select
    upper("Hola amigos") as upper,							/* Convierte una cadena a mayusculas */
    lower("Hola amigos") as lower,							/* Convierte una cadena a minusculas */
    trim("    Hola amigos     ") as trim,						/* Elimina espacios en blanco al inicio y final de una cadena */
    trim(leading "$" from "$$Hola amigos") as trim2, 				/* Elimina un caracter inicial que le especifiquemos de una cadena de texto */
    replace("Hola amigos","Hola","Holi") as reemplazar;				/* Permite sustituir una cadena de texto por otra dentro de una string */
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,upper,lower,trim,trim2,reemplazar
0,HOLA AMIGOS,hola amigos,Hola amigos,Hola amigos,Holi amigos


In [ ]:
# Usar la funcion trim para eliminar espacios en blanco de la columna comentario. 
# Convertir la columna rating a entero sin signo.
# Convertir la columna año a entero sin signo.
# Convertir la columna mes a entero sin signo.
# Reemplazar el nombre de país.

query = """
select
    convert(rating,unsigned int) as rating,
    case
        when año="missing" then null
        else convert(año,unsigned int)
    end as año,
    case
        when mes="missing" then null
        else convert(mes,unsigned int)
    end as mes,
    replace(replace(replace(pais_origen,"United States","US"),"United Kingdom","UK"),"United Arab Emirates","UAE") as pais_origen,
    trim(comentario) as comentario,
    parque
from
    (
    select
        substring_index(substring_index(information,", Year_Month: ",1),"Rating: ",-1) as rating,
        substring_index(substring_index(information,", Reviewer_Location: ",1),", Year_Month: ",-1) as fecha,
        substring_index(substring_index(substring_index(information,", Reviewer_Location: ",1),", Year_Month: ",-1),"-",1) as año,
        substring_index(substring_index(substring_index(information,", Reviewer_Location: ",1),", Year_Month: ",-1),"-",-1) as mes,
        substring_index(substring_index(information,", Review_Text: ",1),", Reviewer_Location: ",-1) as pais_origen,
        substring_index(substring_index(information,", Branch: ",1),", Review_Text: ",-1) as comentario,
        substring_index(information," Branch: ",-1) as parque
    from
        analisistexto.disneyrew
) as a;
"""

df = pd.read_sql_query(text(query), engine)
with pd.option_context(
    'display.max_colwidth', None,
    'display.max_columns', None):
    display(df.head(10))


,rating,año,mes,pais_origen,comentario,parque
0,4,2019.0,4.0,Australia,If you've ever been to Disneyland anywhere you'll find Disneyland Hong Kong very similar in the layout when you walk into main street! It has a very familiar feel. One of the rides its a Small World is absolutely fabulous and worth doing. The day we visited was fairly hot and relatively busy but the queues moved fairly well. Branch: Disneyland_HongKong,Disneyland_HongKong
1,4,2019.0,5.0,Philippines,"Its been a while since d last time we visit HK Disneyland .. Yet, this time we only stay in Tomorrowland .. AKA Marvel land!Now they have Iron Man Experience n d Newly open Ant Man n d Wasp!!Ironman .. Great feature n so Exciting, especially d whole scenery of HK (HK central area to Kowloon)!Antman .. Changed by previous Buzz lightyear! More or less d same, but I'm expecting to have something most!!However, my boys like it!!Space Mountain .. Turns into Star Wars!! This 1 is Great!!!For cast members (staffs) .. Felt bit MINUS point from before!!! Just dun feel like its a Disney brand!! Seems more local like Ocean Park or even worst!!They got no SMILING face, but just wanna u to enter n attraction n leave!!Hello this is supposed to be Happiest Place on Earth brand!! But, just really Dont feel it!!Bakery in Main Street now have more attractive delicacies n Disney theme sweets .. These are Good Points!!Last, they also have Starbucks now inside the theme park!! Branch: Disneyland_HongKong",Disneyland_HongKong
2,4,2019.0,4.0,UAE,"Thanks God it wasn t too hot or too humid when I was visiting the park otherwise it would be a big issue (there is not a lot of shade).I have arrived around 10:30am and left at 6pm. Unfortunately I didn t last until evening parade, but 8.5 hours was too much for me.There is plenty to do and everyone will find something interesting for themselves to enjoy.It wasn t extremely busy and the longest time I had to queue for certain attractions was 45 minutes (which is really not that bad).Although I had an amazing time, I felt a bit underwhelmed with choice of rides and attractions. The park itself is quite small (I was really expecting something grand even the main castle which was closed by the way was quite small).The food options are good, few coffee shops (including Starbucks) and plenty of gift shops. There was no issue with toilets as they are everywhere.All together it was a great day out and I really enjoyed it. Branch: Disneyland_HongKong",Disneyland_HongKong
3,4,2019.0,4.0,Australia,"HK Disneyland is a great compact park. Unfortunately there is quite a bit of maintenance work going on at present so a number of areas are closed off (including the famous castle) If you go midweek, it is not too crowded and certainly no where near as bus as LA Disneyland. We did notice on this visit that prices for food, drinks etc have really gone through the roof so be prepared to pay top dollar for snacks (and avoid the souvenir shops if you can) Regardless, kids will love it. Branch: Disneyland_HongKong",Disneyland_HongKong
4,4,2019.0,4.0,UK,"the location is not in the city, took around 1 hour from Kowlon, my kids like disneyland so much, everything is fine. but its really crowded and hot in Hong Kong Branch: Disneyland_HongKong",Disneyland_HongKong
5,3,2019.0,4.0,Singapore,"Have been to Disney World, Disneyland Anaheim and Tokyo Disneyland but I feel that Disneyland Hong Kong is really too small to be called a Disneyland. It has way too few rides and attractions. Souvenirs, food and even entrance tickets are slightly more expensive than other Disneyland as well. Basically, this park is good only for small children and people who has never been to Disney. The food choices were acceptable, mostly fast food, and not too expensive. Bottled water, however, was VERY expensive but they do have water fountains around for you to refill your water bottles. The parade was pretty good. It was crowded not a problem but what was the problem was the people were just 

In [ ]:
# Con el resultado de la consulta anterior crear una nueva tabla. Esta tabla se llamará disneyclean y estará en el esquema analisistexto.

query1 = """
DROP TABLE IF EXISTS analisistexto.disneyclean
"""

query2 = """
create table analisistexto.disneyclean as 

select
    convert(rating,unsigned int) as rating,
    case
        when año="missing" then null
        else convert(año,unsigned int)
    end as año,
    case
        when mes="missing" then null
        else convert(mes,unsigned int)
    end as mes,
    replace(replace(replace(pais_origen,"United States","US"),"United Kingdom","UK"),"United Arab Emirates","UAE") as pais_origen,
    trim(comentario) as comentario,
    parque
from
    (
    select
        substring_index(substring_index(information,", Year_Month: ",1),"Rating: ",-1) as rating,
        substring_index(substring_index(information,", Reviewer_Location: ",1),", Year_Month: ",-1) as fecha,
        substring_index(substring_index(substring_index(information,", Reviewer_Location: ",1),", Year_Month: ",-1),"-",1) as año,
        substring_index(substring_index(substring_index(information,", Reviewer_Location: ",1),", Year_Month: ",-1),"-",-1) as mes,
        substring_index(substring_index(information,", Review_Text: ",1),", Reviewer_Location: ",-1) as pais_origen,
        substring_index(substring_index(information,", Branch: ",1),", Review_Text: ",-1) as comentario,
        substring_index(information," Branch: ",-1) as parque
    from
        analisistexto.disneyrew
) as a;
"""

with engine.begin() as conn:
    conn.execute(text(query1)) #Ejecutamos cada consulta dentro de la conexión. Esto asegura que ambas consultas se ejecuten en el mismo contexto de transacción.
    conn.execute(text(query2))

In [66]:
# Visualizar las primeras 5 filas de la tabla disneyclean que acabamos de crear

query = """
select
    *
from
    analisistexto.disneyclean;
"""
df = pd.read_sql_query(text(query), engine)
with pd.option_context(
    'display.max_colwidth', None,
    'display.max_columns', None):
    display(df.head())

,rating,año,mes,pais_origen,comentario,parque
0,4,2019.0,4.0,Australia,If you've ever been to Disneyland anywhere you'll find Disneyland Hong Kong very similar in the layout when you walk into main street! It has a very familiar feel. One of the rides its a Small World is absolutely fabulous and worth doing. The day we visited was fairly hot and relatively busy but the queues moved fairly well. Branch: Disneyland_HongKong,Disneyland_HongKong
1,4,2019.0,5.0,Philippines,"Its been a while since d last time we visit HK Disneyland .. Yet, this time we only stay in Tomorrowland .. AKA Marvel land!Now they have Iron Man Experience n d Newly open Ant Man n d Wasp!!Ironman .. Great feature n so Exciting, especially d whole scenery of HK (HK central area to Kowloon)!Antman .. Changed by previous Buzz lightyear! More or less d same, but I'm expecting to have something most!!However, my boys like it!!Space Mountain .. Turns into Star Wars!! This 1 is Great!!!For cast members (staffs) .. Felt bit MINUS point from before!!! Just dun feel like its a Disney brand!! Seems more local like Ocean Park or even worst!!They got no SMILING face, but just wanna u to enter n attraction n leave!!Hello this is supposed to be Happiest Place on Earth brand!! But, just really Dont feel it!!Bakery in Main Street now have more attractive delicacies n Disney theme sweets .. These are Good Points!!Last, they also have Starbucks now inside the theme park!! Branch: Disneyland_HongKong",Disneyland_HongKong
2,4,2019.0,4.0,UAE,"Thanks God it wasn t too hot or too humid when I was visiting the park otherwise it would be a big issue (there is not a lot of shade).I have arrived around 10:30am and left at 6pm. Unfortunately I didn t last until evening parade, but 8.5 hours was too much for me.There is plenty to do and everyone will find something interesting for themselves to enjoy.It wasn t extremely busy and the longest time I had to queue for certain attractions was 45 minutes (which is really not that bad).Although I had an amazing time, I felt a bit underwhelmed with choice of rides and attractions. The park itself is quite small (I was really expecting something grand even the main castle which was closed by the way was quite small).The food options are good, few coffee shops (including Starbucks) and plenty of gift shops. There was no issue with toilets as they are everywhere.All together it was a great day out and I really enjoyed it. Branch: Disneyland_HongKong",Disneyland_HongKong
3,4,2019.0,4.0,Australia,"HK Disneyland is a great compact park. Unfortunately there is quite a bit of maintenance work going on at present so a number of areas are closed off (including the famous castle) If you go midweek, it is not too crowded and certainly no where near as bus as LA Disneyland. We did notice on this visit that prices for food, drinks etc have really gone through the roof so be prepared to pay top dollar for snacks (and avoid the souvenir shops if you can) Regardless, kids will love it. Branch: Disneyland_HongKong",Disneyland_HongKong
4,4,2019.0,4.0,UK,"the location is not in the city, took around 1 hour from Kowlon, my kids like disneyland so much, everything is fine. but its really crowded and hot in Hong Kong Branch: Disneyland_HongKong",Disneyland_HongKong


##### 5-9 Busquedas con Comodin

In [ ]:
# Funcion LIKE

query = """
select
    "Daniel Perez Gonzalez" LIKE "Go" as e1,      /* Devuelve 0. Busca coincidencia exacta entre las dos cadenas.*/
    "Daniel Perez Gonzalez" NOT LIKE "Go" as e2,  /* Devuelve 1. Busca la NO coincidencia exacta entre las dos cadenas.*/
    "Daniel Perez Gonzalez" LIKE "%on%" as e3,    /* Devuelve 1. Busca que la cadena empiece con texto cualquiera seguida de la cadena "on" seguida de texto cualquiera.*/
    "Daniel Perez Gonzalez" LIKE "d%" as e4,      /* Devuelve 1. Busca que la cadena empiece con la palabra "d" seguida de texto cualquiera.*/
    "Daniel Perez Gonzalez" LIKE "P%" as e5,      /* Devuelve 0. Busca que la cadena empiece con la palabra "P" seguida de texto cualquiera.*/
    "Daniel Perez Gonzalez" LIKE "%Z" as e6,      /* Devuelve 1. Busca que la cadena empiece con texto cualquiera y termine con "z".*/
    "Daniel Perez Gonzalez" LIKE "%el" as e7,     /* Devuelve 0. Busca que la cadena empiece con texto cualquiera y termine con "el".*/
    "Daniel Perez Gonzalez" LIKE "_a%" as e8,     /* Devuelve 1. Busca que la cadena empiece con un caracter cualquiera, seguida de la letra "a" seguido de texto cualquiera.*/
    "Daniel Perez Gonzalez" LIKE "_e%" as e9;     /* Devuelve 0. Busca que la cadena empiece con un caracter cualquiera, seguida de la letra "e" seguido de texto cualquiera.*/
"""
df = pd.read_sql_query(text(query), engine)
df.head(10)

,e1,e2,e3,e4,e5,e6,e7,e8,e9
0,0,1,1,1,0,1,0,1,0
